In [20]:
import ee
import sys
import random

# Initialize Earth Engine
EE_PROJECT = 'arctic-biomass-mapping'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# ====================
# 1. SET UP ==========
# ====================

# 1.0 PARAMETERS ==========

fire_size_thresh_acre = 10
version = 'v20260512'
fire_start_yr = 1940
fire_end_yr = 2025
scale = 30

# 1.1 ROI DATA ==========

roi = ee.FeatureCollection("projects/fisl-tundra-fire/assets/rois/above_arctic_simple")
roi_mask = ee.Image('projects/foreststructure/ABoVE/CCDC/CCDC_ABoVE_L4578SR_1984_2020_J091_273_v20201203_mappedArea_na_beringia_westCan')

# 1.2 FIRE DATA ==========

fires_historic = ee.ImageCollection("projects/fisl-tundra-fire/assets/fires_raster/akca_fires_30m_3338_arctic")
fires_modern = ee.ImageCollection('projects/fisl-tundra-fire/assets/potter_fire_v20260430_ingested')

# Tidy historic data
fires_historic = fires_historic.map(lambda img: img.set('year', ee.Number.parse(img.get('year'))))
fires_historic = fires_historic.filter(ee.Filter.Or(ee.Filter.lt('year', 2001), ee.Filter.eq('year', 2025)))

# Tidy modern data
fires_modern = fires_modern.select('Burn_Mask')
fires_modern_years = fires_modern.aggregate_array('year').distinct()

def process_modern(yr):
    fires_modern_filter = fires_modern.filter(ee.Filter.eq('year', yr))
    return fires_modern_filter.max() \
        .selfMask() \
        .multiply(ee.Image.constant(yr)) \
        .uint16() \
        .copyProperties(fires_modern_filter.first())

fires_modern = ee.ImageCollection(fires_modern_years.map(process_modern))

# Combine
fires = fires_historic.merge(fires_modern)
fire_years_all = fires.aggregate_array('year').distinct().sort().getInfo()

def add_date_info(yr):
    img = fires.filter(ee.Filter.eq('year', yr)).first()
    next_yr = ee.Number(yr).add(1)
    return img.set({
        'system:time_start': ee.Date.fromYMD(yr, 1, 1).millis(),
        'system:time_end': ee.Date.fromYMD(next_yr, 1, 1).millis()
    }).rename('fire_year')

fires = ee.ImageCollection(ee.List(fire_years_all).map(add_date_info))

# Restrict to desired fire years
fire_years = ee.List.sequence(fire_start_yr, fire_end_yr).filter(ee.Filter.inList('item', fire_years_all))
fire_years_list = fire_years.getInfo() # Get to client side for Python loop

# 1.3 LAND COVER DATA ==========

#  Macander: Before 1984, an experimental land cover products based on MSS CCDC I created
lc_1972_1984 = ee.ImageCollection('projects/foreststructure/Alaska/MSS/above_lc_mss_v20220619_tests') \
    .filter(ee.Filter.calendarRange(1972, 1984, 'year')) \
    .toBands() \
    .regexpRename('_bootstrap_20220619_mss_ccdc_1985_1986_n5000_2000_500_seed6_rf100_ml2_above_lc_co', '') \
    .regexpRename('above_lc_', 'y')

# Wang et al. land cover 1985-2014
lc_1985_2014 = ee.Image("projects/foreststructure/ABoVE/ORNL_DAAC/ABoVE_LandCover_v01")

# Macander: After 2014, using standard CCDC (probably from the same segments/coefficients used on the PFT mapping) trained on the 1985-2014 map
lc_2015_2020 = ee.ImageCollection("projects/foreststructure/ABoVE/BiomeShift/ABoVE_LC_2020") \
    .select('y2015','y2016','y2017','y2018','y2019','y2020') \
    .mosaic()

# Combine
lc_img = lc_1972_1984.addBands(lc_1985_2014).addBands(lc_2015_2020)

# Add system start and end times
lc_years_all = ee.List.sequence(1972, 2020).getInfo()
def process_lc(year):
    band = 'y' + str(year)
    next_year = year + 1
    return lc_img.select(band) \
        .set({
            'system:time_start': ee.Date.fromYMD(year, 1, 1).millis(),
            'system:time_end': ee.Date.fromYMD(next_year, 1, 1).millis(),
            'year': year
        }).rename('lc')
lc = ee.ImageCollection([process_lc(y) for y in lc_years_all])

# 1.4 PFT DATA ==========

# Get PFT data
pft = ee.ImageCollection("projects/foreststructure/ABoVE/BiomeShift/Alaska_Yukon_PFT_202207_Filled")

# Create function to tidy PFT data for a specific year
def getPftYearStack(pft_ic, year):
    pft_year_coll = pft_ic.filter(ee.Filter.calendarRange(year, None, 'year')).select('cover')
    
    responses = [
        'cTree', 'bTree', 'allDecShrub', 'decshrabs', 'alnshr', 'betshr', 
        'salshr', 'allEvShrub', 'graminoid', 'allForb', 'tmlichenLight2', 'talshr'
    ]
    
    stack = None
    for res in responses:
        band_img = pft_year_coll.filterMetadata('response', 'equals', res).first().rename(res + '_cover')
        if stack is None:
            stack = band_img
        else:
            stack = stack.addBands(band_img)
            
    stack = stack.clamp(0, 100)
    pft_year_mask = stack.mask().reduce(ee.Reducer.min())
    return stack.updateMask(pft_year_mask)

# =============================================
# 2. EXTRACT DATA ACROSS FIRE YEARS ===========
# =============================================

# TEMP +++++++++++++++++++++++++++++++++++
fire_years_list = [1988]
# TEMP +++++++++++++++++++++++++++++++++++

for fire_year in fire_years_list:
    
    # 2.1 GET AND TIDY CURRENT YEAR FIRES ==========

    # Get current fire year fires
    fire_year_img = fires.filter(ee.Filter.calendarRange(fire_year, fire_year, 'year')).first()
    
    # Create closest previous fire year image
    # Get all pre-fire years only (per current 'fire_year')
    # Take the max to create a 'closest previous fire year' mosaic
    # All unburned areas get assigned 1900 as the most recent fire year
    if fire_year != fire_start_yr:
        prevfire_year = fires.filter(ee.Filter.calendarRange(1900, fire_year - 1, 'year')) \
            .max().unmask(1900).rename('year')
    else:
        prevfire_year = ee.Image(1900).rename('year')
    
    # Create closest next fire year image
    # Get all post-fire years only (per current 'fire_year')
    # Take the min to create an 'closest next fire year' mosaic
    # All unburned areas get assigned 2030 as the oldest fire year
    if fire_year != fire_end_yr:
        nextfire_year = fires.filter(ee.Filter.calendarRange(fire_year + 1, 2030, 'year')) \
            .min().unmask(2030).rename('year')
    else:
        nextfire_year = ee.Image(2030).rename('year')
   
    # 2.2 GET PRE-FIRE LAND COVER DATA ==========

    # Get land cover data for the year before fire
    prefire_year = fire_year - 1
    if prefire_year < 1972 or prefire_year > 2020:
        prefire_lc = ee.Image(0).rename('lc')
    else:
        prefire_lc = lc.filter(ee.Filter.calendarRange(prefire_year, prefire_year, 'year')) \
            .first().select('lc').unmask(0).multiply(10)
        
    # 2.2 EXTRACT DATA ACROSS PFT YEARS ==========
    
    # Initialize holding list and year sequence
    pft_fc_list = []
    pft_years = ee.List.sequence(1985, 2020, 1).getInfo()
    
    for pft_year in pft_years:
        
        # 2.3 SET UP PFT, FIRE, AND LAND COVER DATA ==========

        # Get current PFT year PFT data
        pft_year_stack = getPftYearStack(pft, pft_year)
        
        # Mask fire data to exclude previous or subsequent fires that disrupt PFT trajectory
        fire_history_mask = ee.Image(pft_year).gt(prevfire_year).And(ee.Image(pft_year).lt(nextfire_year))
        fire_year_img_w_history = fire_year_img.updateMask(fire_history_mask)
               
        # Stack data
        stack = prefire_lc.addBands(pft_year_stack) \
            .updateMask(fire_year_img_w_history) \
            .updateMask(roi_mask)
        
        # Specify PFT names and percentile labels
        pftNames = pft_year_stack.bandNames()
        percentileLabels = ['p2', 'p3', 'p50', 'p97', 'p98']
        
        # 2.4 SUMMARIZE PFT COVER ACROSS LAND COVER TYPES ==========
        # Group by prefire land cover
        # Calculate median and confidence intervals per PFT
        
        crosstab = stack.reduceRegion(
            reducer=ee.Reducer.percentile([2, 3, 50, 97, 98]).unweighted().repeat(pftNames.length()).group(0),
            geometry=roi.geometry(),
            scale=scale,
            maxPixels=1e12
        )
        
        # 2.5 TIDY ==========
        
        # Define function to turn messy dictionary into legible feature collection
        def flatten_logic(groupObj):
            groupObj = ee.Dictionary(groupObj)
            lcCode = groupObj.get('group')
            
            def map_percentiles(pLabel):
                pValues = ee.List(groupObj.get(pLabel))
                
                def map_pfts(pftName):
                    pftIndex = pftNames.indexOf(pftName)
                    value = pValues.get(pftIndex)
                    return ee.Feature(None, {
                        'land_cover': lcCode,
                        'percentile': pLabel,
                        'pft_type': pftName,
                        'cover_value': value,
                        'pft_year': pft_year,
                        'fire_year': fire_year
                    })
                return pftNames.map(map_pfts)
            return ee.List(percentileLabels).map(map_percentiles)

        # Apply function to tidy to legible feature collection
        groups = ee.List(crosstab.get('groups'))
        final_fc = ee.FeatureCollection(groups.map(flatten_logic).flatten())
        
        # Add to list of pft_year feature collections
        pft_fc_list.append(final_fc)

    # 2.6 AGGREGATE DATA ACROSS PFT YEARS ==========

    # Convert list of FCs into one nested FC and flatten
    # This turns [FC1, FC2, FC3] into one big FC
    final_fire_year_fc = ee.FeatureCollection(pft_fc_list).flatten()
    
    # Add fire_year property to everything
    final_fire_year_fc = final_fire_year_fc.map(lambda f: f.set('fire_year', fire_year))

    # 2.7 EXPORT ==========
    
    task = ee.batch.Export.table.toDrive(
        collection=final_fire_year_fc,
        description=f'lc_pft_fire{fire_year}_{version}',
        folder='pft_trajectories',
        fileFormat='CSV'
    )
    
#     task.start()
    print(f"Aggregated {len(pft_fc_list)} PFT years for fire year {fire_year}. Task ready.")

Aggregated 36 PFT years for fire year 1988. Task ready.
